# 14 Data Warehouse — Load the Modeled Star Schema

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. DWH boundary

The DWH contains modeled analytical structures, not raw source representations.

## 2. Build DuckDB warehouse

DuckDB provides a lightweight local warehouse for classroom use.

In [ ]:
import duckdb
con=duckdb.connect('data/icecream_dw.duckdb')
for table in ['dim_customer','dim_product','dim_store','dim_date','fact_sales']:
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv_auto('data/{table}.csv')")
print(con.sql("SHOW TABLES"))

## 3. Query warehouse

Use SQL to answer analytical questions after modeling.

In [ ]:
print(con.sql("SELECT p.product_name, SUM(f.sales_amount) total_sales FROM fact_sales f JOIN dim_product p ON f.product_key=p.product_key GROUP BY p.product_name ORDER BY total_sales DESC LIMIT 10"))

## 4. Reconcile warehouse

Compare warehouse totals with staged source totals.

In [ ]:
source=pd.read_csv('../11_staging_layer/data/stg_sales.csv')
dw_total=con.sql('SELECT SUM(sales_amount) FROM fact_sales').fetchone()[0]
print('source:',source.sales_amount.sum(),'warehouse:',dw_total)
assert abs(source.sales_amount.sum()-dw_total)<0.01

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.